In [118]:
# 📦 Imports
import pandas as pd
import json
from tqdm import tqdm
from textblob import TextBlob

In [119]:
# 1. Load Real Amazon Beauty Reviews
# -------------------------------
with open("All_Beauty_5.json", "r") as f:
    beauty_data = [json.loads(line) for line in f]

df_beauty = pd.DataFrame(beauty_data)
df_beauty_real = df_beauty[
    (df_beauty['verified'] == True) &
    (df_beauty['reviewText'].str.len() > 50)
].copy()

df_beauty_real["label"] = 0
df_beauty_real["category"] = "beauty"
df_beauty_real["source"] = "amazon"
df_beauty_real["stars"] = df_beauty_real["overall"]
df_beauty_real["verified"] = True
df_beauty_real["generated"] = False

df_beauty_real = df_beauty_real[["reviewText", "label", "source", "category", "stars", "verified", "generated"]]

In [120]:
# 2. Load Real Amazon Electronics Reviews
# -------------------------------
filtered_reviews = []
MAX_REVIEWS = 4000
with open("Electronics_5.json", "r") as f:
    for line in f:
        review = json.loads(line)
        if (
            review.get("verified") and
            len(review.get("reviewText", "")) > 50 and
            review.get("overall") is not None
        ):
            filtered_reviews.append({
                "reviewText": review["reviewText"],
                "label": 0,
                "source": "amazon",
                "category": "electronics",
                "stars": review["overall"],
                "verified": True,
                "generated": False
            })
            if len(filtered_reviews) >= MAX_REVIEWS:
                break

df_electronics_real = pd.DataFrame(filtered_reviews)

In [121]:
df_electronics_real.head()

,reviewText,label,source,category,stars,verified,generated
0,This is the best novel I have read in 2 or 3 y...,0,amazon,electronics,5.0,True,False
1,"Pages and pages of introspection, in the style...",0,amazon,electronics,3.0,True,False
2,I was taken in by reviews that compared this b...,0,amazon,electronics,3.0,True,False
3,I read this probably 50 years ago in my youth ...,0,amazon,electronics,4.0,True,False
4,I read every Perry mason book voraciously. Fin...,0,amazon,electronics,5.0,True,False


In [122]:
# Load and normalize crawl-stage dataset
df_crawl = pd.read_csv("fake_reviews_dataset.csv")

# Rename columns to match unified schema
df_crawl = df_crawl.rename(columns={
    "text_": "reviewText",
    "rating": "stars"
})

# Map labels: 'CG' = 1 (fake), 'OR' = 0 (real)
df_crawl["label"] = df_crawl["label"].map({"CG": 1, "OR": 0})

# Add metadata
df_crawl["source"] = "crawl_stage"
df_crawl["verified"] = False
df_crawl["generated"] = False

# Ensure category column exists and clean missing values
if "category" not in df_crawl.columns:
    df_crawl["category"] = "unknown"
else:
    df_crawl["category"] = df_crawl["category"].fillna("unknown")

# Ensure stars are numeric and clamp between 1 and 5
df_crawl["stars"] = pd.to_numeric(df_crawl["stars"], errors='coerce').clip(1, 5).fillna(3)

# Keep only relevant columns
df_crawl = df_crawl[["reviewText", "label", "source", "category", "stars", "verified", "generated"]]


In [123]:
# Map verbose Amazon-like categories to simple ones
category_map = {
    "Home_and_Kitchen_5": "home",
    "Sports_and_Outdoors_5": "sports",
    "Electronics_5": "electronics",
    "Movies_and_TV_5": "movies",
    "Tools_and_Home_Improvement_5": "tools",
    "Pet_Supplies_5": "pets",
    "Kindle_Store_5": "ebooks",
    "Books_5": "books",
    "Toys_and_Games_5": "toys",
    "Clothing_Shoes_and_Jewelry_5": "fashion"
}

# Apply to df_crawl
df_crawl["category"] = df_crawl["category"].map(category_map)


In [124]:
df_crawl.head()

,reviewText,label,source,category,stars,verified,generated
0,"Love this! Well made, sturdy, and very comfor...",1,crawl_stage,home,5.0,False,False
1,"love it, a great upgrade from the original. I...",1,crawl_stage,home,5.0,False,False
2,This pillow saved my back. I love the look and...,1,crawl_stage,home,5.0,False,False
3,"Missing information on how to use it, but it i...",1,crawl_stage,home,1.0,False,False
4,Very nice set. Good quality. We have had the s...,1,crawl_stage,home,5.0,False,False


In [125]:
# List of LLM-generated fake review files ONLY
llm_files = [
    "Beauty_Enthousiaste_ChatGPT_100.csv",
    "Beauty_Rage_1star_ChatGPT_50.csv",
    "Beauty_Realiste_Claude_100.csv",
    "Beauty_Robotique_Deepseek_50.csv",
    "Electronics_realiste_Claude_100.csv",
    "Electronics_Robotique_Deepseek_50_avis_FIXED.csv",
    "Electronics-Enthousiaste-ChatGPT-100-avis.csv",
    "Electronics_Rage_1star_ChatGPT_50_.csv"
]

# Storage for all loaded LLM-generated fake reviews
df_llm = []

for file in llm_files:
    df = pd.read_csv(file)

    # Normalize columns
    if "reviewText" not in df.columns and "text" in df.columns:
        df.rename(columns={"text": "reviewText"}, inplace=True)
    if "star" in df.columns:
        df.rename(columns={"star": "stars"}, inplace=True)

    # Add missing fields
    df["label"] = 1
    df["verified"] = False
    df["generated"] = True

    # Add category based on filename
    if "Beauty" in file:
        df["category"] = "beauty"
    elif "Electronics" in file:
        df["category"] = "electronics"
    else:
        df["category"] = "unknown"

    # Assign stars if missing using sentiment
    if "stars" not in df.columns or df["stars"].isnull().all():
        df["stars"] = df["reviewText"].apply(
            lambda x: round((TextBlob(str(x)).sentiment.polarity + 1) * 2.5)
        )

    # Final structure
    df = df[["reviewText", "label", "source", "category", "stars", "verified", "generated"]] if "source" in df.columns else df.assign(source="llm")[["reviewText", "label", "source", "category", "stars", "verified", "generated"]]

    df_llm.append(df)

# Combine all
df_llm_combined = pd.concat(df_llm, ignore_index=True)

# ✅ Preview
df_llm_combined.sample(3)

,reviewText,label,source,category,stars,verified,generated
535,🔋 🔥 👌 🚀 This wireless earbuds is absolutely in...,1,llm,electronics,5,False,True
549,👍 📱 This Bluetooth speaker is absolutely fanta...,1,llm,electronics,5,False,True
407,The product functions properly. It meets requi...,1,llm,electronics,4,False,True


In [126]:
# 5. Load YelpZIP (~10,000 reviews)
# -------------------------------
df_yelp = pd.read_csv("yelpzip.csv")
df_yelp = df_yelp.rename(columns={"text": "reviewText", "rating": "stars"})
df_yelp = df_yelp[df_yelp["reviewText"].str.len() > 50].sample(n=10000, random_state=42)

df_yelp["label"] = df_yelp["label"].map({1: 0, -1: 1})
df_yelp["category"] = "restaurant"
df_yelp["source"] = "yelpzip"
df_yelp["verified"] = True
df_yelp["generated"] = False

df_yelp = df_yelp[["reviewText", "label", "source", "category", "stars", "verified", "generated"]]


In [127]:
# 6. Load Ott Dataset
# -------------------------------
df_ott = pd.read_csv("deceptive-opinion.csv")
df_ott = df_ott.rename(columns={"text": "reviewText"})
df_ott["label"] = df_ott["deceptive"].map({"truthful": 0, "deceptive": 1})
df_ott["category"] = "hotel"
df_ott["source"] = "ott"
df_ott["stars"] = 3.0
df_ott["verified"] = False
df_ott["generated"] = False

df_ott = df_ott[["reviewText", "label", "source", "category", "stars", "verified", "generated"]]


In [128]:
# 7. Combine All Sources
# -------------------------------
df_all = pd.concat([
    df_beauty_real,
    df_electronics_real,
    df_crawl,
    df_llm_combined,
    df_yelp,
    df_ott
], ignore_index=True)

In [129]:
# 8. Add Metadata (sentiment + length)
# -------------------------------
df_all["review_length"] = df_all["reviewText"].apply(lambda x: len(x.split()))
df_all["sentiment"] = df_all["reviewText"].apply(lambda x: round(TextBlob(str(x)).sentiment.polarity, 3))


In [130]:
df_all.head(50)
df_all.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 59528 entries, 0 to 59527
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   reviewText     59528 non-null  object 
 1   label          59528 non-null  int64  
 2   source         59528 non-null  object 
 3   category       59528 non-null  object 
 4   stars          59528 non-null  float64
 5   verified       59528 non-null  bool   
 6   generated      59528 non-null  bool   
 7   review_length  59528 non-null  int64  
 8   sentiment      59528 non-null  float64
dtypes: bool(2), float64(2), int64(2), object(3)
memory usage: 3.3+ MB


In [131]:
# 🧠 Overview of Run Stage Dataset
print("✅ Dataset Ready for Run Stage")
print(f"Total reviews: {df_all.shape[0]}")
print("\nLabel distribution:\n", df_all["label"].value_counts())
print("\nCategory distribution:\n", df_all["category"].value_counts())
print("\nSource distribution:\n", df_all["source"].value_counts())
print("\nSample rows:")
display(df_all.sample(5))


✅ Dataset Ready for Run Stage
Total reviews: 59528

Label distribution:
 label
0    36563
1    22965
Name: count, dtype: int64

Category distribution:
 category
restaurant     10000
electronics     8291
ebooks          4730
books           4370
pets            4254
home            4056
sports          3946
tools           3858
fashion         3848
toys            3794
movies          3588
beauty          3193
hotel           1600
Name: count, dtype: int64

Source distribution:
 source
crawl_stage    40432
yelpzip        10000
amazon          6892
ott             1600
llm              604
Name: count, dtype: int64

Sample rows:


,reviewText,label,source,category,stars,verified,generated,review_length,sentiment
39928,Love the fish to go with it and the wooden par...,1,crawl_stage,toys,5.0,False,False,19,0.275
21803,"I've played three movies thus far, on my Sony ...",0,crawl_stage,movies,2.0,False,False,149,0.103
2713,"I received my order on time, and the products ...",0,amazon,beauty,4.0,True,False,25,-0.050
44004,"Waterville holder on tube source, has the wide...",1,crawl_stage,fashion,3.0,False,False,12,0.450
29327,I was recently informed by a local pet store t...,1,crawl_stage,pets,1.0,False,False,64,0.190


In [132]:
print("Stars value counts:\n", df_all["stars"].value_counts())

Stars value counts:
 stars
5.0    33652
4.0    12501
3.0     7177
1.0     3285
2.0     2913
Name: count, dtype: int64


In [133]:
print("Missing values per column:\n", df_all.isnull().sum())


Missing values per column:
 reviewText       0
label            0
source           0
category         0
stars            0
verified         0
generated        0
review_length    0
sentiment        0
dtype: int64


In [134]:
df_all["length"] = df_all["reviewText"].apply(lambda x: len(str(x).split()))
print(df_all["length"].describe())


count    59528.000000
mean        75.533850
std         81.289936
min          1.000000
25%         22.000000
50%         44.000000
75%         98.000000
max       3588.000000
Name: length, dtype: float64


In [135]:
# 9. Save
# -------------------------------
df_all = df_all.dropna(subset=["reviewText"])
df_all.to_csv("run_stage_dataset.csv", index=False)

print("✅ Dataset created and saved as run_stage_dataset.csv")
print(df_all["label"].value_counts())
print(df_all["category"].value_counts())

✅ Dataset created and saved as run_stage_dataset.csv
label
0    36563
1    22965
Name: count, dtype: int64
category
restaurant     10000
electronics     8291
ebooks          4730
books           4370
pets            4254
home            4056
sports          3946
tools           3858
fashion         3848
toys            3794
movies          3588
beauty          3193
hotel           1600
Name: count, dtype: int64
